In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 129.8 MB/s  0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.15 requires cellxgene-census==1.15.0, which is not installed.
pytdc 1.1.15 requires dataclasses<1.0,>=0.6, which is not installed.
pytdc 1.1.15 requires evaluate==0.4.2, which is not installed.
pytdc 1.1.15 requires gget<1.0.0,>=0.28.4, which is not installed.
pytdc 1.1.15 requires tiledbsoma<2.0.0,>=1.7.2, which is not installed.
pytdc 1.1.15 requires accelerate==0.33.0, but you have accelerate 1.13.0 which is incompatible.
pytdc 1.1.15 requires datasets<2.20.0, but you have datasets 4.0.0 which is incompatible.
pytdc 1.1.15 requires huggingface_hub<1.0,>=0.20.3, but you have huggingface-hub 1.10.1 which is incompatible.
pytdc 1.1.15 requires numpy<2.0.0,>=1.26.4, but you have numpy 2.4.4 which is incompatible.
pytdc 1.1.15 req

In [ ]:
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 62.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 60.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 38.5 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pand

  Using cached pytdc-1.1.15.tar.gz (154 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pytdc: filename=pytdc-1.1.15-py3-none-any.whl size=191725 sha256=73e6b5d11e43d774c7139ad4c247b4235160a0eda2eaf34ea78f1e45facdac2b
  Stored in directory: /root/.cache/pip/wheels/3a/51/0f/0b00fd8b8288143eec76ea0a57804cddd72aae207cc4cb4d65
Successfully built pytdc


In [ ]:
!pip install fuzzywuzzy

  Using cached fuzzywuzzy-0.18.0-py2.py3-none-any.whl.metadata (4.9 kB)
Using cached fuzzywuzzy-0.18.0-py2.py3-none-any.whl (18 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.15 requires cellxgene-census==1.15.0, which is not installed.
pytdc 1.1.15 requires dataclasses<1.0,>=0.6, which is not installed.
pytdc 1.1.15 requires evaluate==0.4.2, which is not installed.
pytdc 1.1.15 requires gget<1.0.0,>=0.28.4, which is not installed.
pytdc 1.1.15 requires rdkit<2024.3.1,>=2023.9.5, which is not installed.
pytdc 1.1.15 requires tiledbsoma<2.0.0,>=1.7.2, which is not installed.
pytdc 1.1.15 requires accelerate==0.33.0, but you have accelerate 1.13.0 which is incompatible.
pytdc 1.1.15 requires datasets<2.20.0, but you have datasets 4.0.0 which is incompatible.
pytdc 1.1.15 requires huggingface_hub<1.0,>=0.20.3, but you have huggingface-hub 1.10.1 whi

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. DEFINICJE KLAS (Architektura i Featurizer)
# ==========================================

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))

  #========================================================================================================================

class Featurizer:
    def __init__(self, y_column, smiles_col='Drug', **kwargs):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.__dict__.update(kwargs)
    def __call__(self, df):
        raise NotImplementedError()

class ECFPFeaturizer(Featurizer):
    def __init__(self, y_column, radius=2, length=1024, **kwargs):
        self.radius = radius
        self.length = length
        super().__init__(y_column, **kwargs)

    def __call__(self, df):
        fingerprints = []
        labels = []
        for i, row in df.iterrows():
            y = row[self.y_column]
            smiles = row[self.smiles_col]
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.length)
                fingerprints.append(np.array(fp))
                labels.append(y)
        return np.array(fingerprints), np.array(labels).reshape(-1, 1)



DLA FINGERPRINTÓW

In [ ]:
# ==========================================
# 2. GŁÓWNY POTOK OBLICZENIOWY
# ==========================================

# A. Pobranie danych
print("[INFO] Pobieranie danych Caco-2 z TDC...")
data = ADME(name='Caco2_Wang')
split = data.get_split()
train_df, test_df = split['train'], split['test']
print(f"[OK] Załadowano dane. Trening: {len(train_df)}, Test: {len(test_df)}")

# B. Przetworzenie na fingerprinty
print(f"[INFO] Generowanie fingerprintów (ECFP, length=1024)...")
featurizer = ECFPFeaturizer(y_column='Y', length=1024)
X_train_np, y_train_np = featurizer(train_df)
X_test_np, y_test_np = featurizer(test_df)
print("[OK] Fingerprinty gotowe.")

#================================================================================================================

# C. Normalizacja etykiet
print("[INFO] Normalizacja danych wyjściowych...")
scaler = StandardScaler()
y_train_scaled = scaler.fit_transform(y_train_np)
y_test_scaled = scaler.transform(y_test_np)

# D. Konwersja na tensory
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_t = torch.FloatTensor(X_train_np).to(device)
y_train_t = torch.FloatTensor(y_train_scaled).to(device)
X_test_t = torch.FloatTensor(X_test_np).to(device)
print(f"[INFO] Urządzenie obliczeniowe: {device}")

# E. Trening
input_dim = featurizer.length
model = STL_NN_Regressor(input_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\n--- START TRENINGU ---")
model.train()
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"  Epoka {epoch:3d} | Błąd MSE: {loss.item():.4f}")

# F. Ewaluacja
print("\n--- EWALUACJA ---")
model.eval()
with torch.no_grad():
    preds_scaled = model(X_test_t).cpu().numpy()
    preds = scaler.inverse_transform(preds_scaled)

    # Obliczenie RMSE
    rmse = np.sqrt(mean_squared_error(y_test_np, preds))

    print(f"Wynik końcowy RMSE dla Caco-2: {rmse:.4f}")
    print("\nPorównanie predykcji z rzeczywistością (pierwsze 5 próbek):")
    for i in range(5):
        print(f"  Prawda: {y_test_np[i][0]:.4f} | Predykcja: {preds[i][0]:.4f}")

print("\n--- KONIEC ---")

Found local copy...
Loading...
Done!


[INFO] Pobieranie danych Caco-2 z TDC...
[OK] Załadowano dane. Trening: 637, Test: 182
[INFO] Generowanie fingerprintów (ECFP, length=1024)...


[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerator
[08:58:20] DEPRECATION WARNING: please use MorganGenerat

[OK] Fingerprinty gotowe.
[INFO] Normalizacja danych wyjściowych...
[INFO] Urządzenie obliczeniowe: cuda

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.1253
  Epoka  20 | Błąd MSE: 0.1986
  Epoka  40 | Błąd MSE: 0.0788
  Epoka  60 | Błąd MSE: 0.0390
  Epoka  80 | Błąd MSE: 0.0281

--- EWALUACJA ---
Wynik końcowy RMSE dla Caco-2: 0.5032

Porównanie predykcji z rzeczywistością (pierwsze 5 próbek):
  Prawda: -5.4891 | Predykcja: -5.6259
  Prawda: -4.8499 | Predykcja: -4.7226
  Prawda: -3.9201 | Predykcja: -4.9417
  Prawda: -5.6720 | Predykcja: -5.5369
  Prawda: -4.9900 | Predykcja: -5.3054

--- KONIEC ---


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score
# ==========================================
# 2. GŁÓWNY POTOK (HIA_Hou)
# ==========================================

print("--- START: KLASYFIKACJA HIA_Hou ---")

# A. Pobranie danych
print("[INFO] Pobieranie danych HIA_Hou z TDC...")
data = ADME(name='HIA_Hou')
split = data.get_split()
train_df, test_df = split['train'], split['test']

# B. Generowanie fingerprintów
print("[INFO] Generowanie fingerprintów...")
featurizer = ECFPFeaturizer(y_column='Y', length=1024)
X_train_np, y_train_np = featurizer(train_df)
X_test_np, y_test_np = featurizer(test_df)

# C. Konwersja na tensory (w klasyfikacji NIE skalujemy Y!)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_t = torch.FloatTensor(X_train_np).to(device)
y_train_t = torch.FloatTensor(y_train_np).to(device) # Etykiety 0.0 lub 1.0
X_test_t = torch.FloatTensor(X_test_np).to(device)

# D. Model i Trening
model = STL_NN_Classifier(input_dim=1024).to(device)
criterion = nn.BCELoss() # Binary Cross Entropy - standard dla klasyfikacji
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\n--- START TRENINGU (Klasyfikacja) ---")
model.train()
for epoch in range(101):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"  Epoka {epoch:3d} | Błąd BCE Loss: {loss.item():.4f}")

# E. Ewaluacja
print("\n--- EWALUACJA ---")
model.eval()
with torch.no_grad():
    # Model zwraca prawdopodobieństwo (0 do 1)
    probs = model(X_test_t).cpu().numpy()
    # Klasy (0 lub 1) ustalamy progiem 0.5
    preds = (probs > 0.5).astype(int)

    # Obliczamy AUROC - kluczowa metryka dla klasyfikacji ADMET
    auroc = roc_auc_score(y_test_np, probs)
    acc = accuracy_score(y_test_np, preds)

    print(f"Wynik końcowy AUROC (HIA): {auroc:.4f}")
    print(f"Dokładność (Accuracy): {acc*100:.2f}%")

    print("\nPrzykładowe wyniki (pierwsze 5):")
    for i in range(5):
        stan = "Wchłania się" if y_test_np[i][0] == 1 else "Brak wchłaniania"
        print(f"  Prawda: {y_test_np[i][0]} ({stan}) | Prawdopodobieństwo: {probs[i][0]:.4f}")

print("\n--- KONIEC ---")

Found local copy...
Loading...
Done!


--- START: KLASYFIKACJA HIA_Hou ---
[INFO] Pobieranie danych HIA_Hou z TDC...
[INFO] Generowanie fingerprintów...


[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerator
[08:55:31] DEPRECATION WARNING: please use MorganGenerat


--- START TRENINGU (Klasyfikacja) ---
  Epoka   0 | Błąd BCE Loss: 1.0869
  Epoka  20 | Błąd BCE Loss: 0.0253
  Epoka  40 | Błąd BCE Loss: 0.0007
  Epoka  60 | Błąd BCE Loss: 0.0002
  Epoka  80 | Błąd BCE Loss: 0.0001
  Epoka 100 | Błąd BCE Loss: 0.0001

--- EWALUACJA ---
Wynik końcowy AUROC (HIA): 0.8757
Dokładność (Accuracy): 92.24%

Przykładowe wyniki (pierwsze 5):
  Prawda: 1 (Wchłania się) | Prawdopodobieństwo: 0.9998
  Prawda: 0 (Brak wchłaniania) | Prawdopodobieństwo: 0.9993
  Prawda: 1 (Wchłania się) | Prawdopodobieństwo: 1.0000
  Prawda: 1 (Wchłania się) | Prawdopodobieństwo: 0.9883
  Prawda: 1 (Wchłania się) | Prawdopodobieństwo: 1.0000

--- KONIEC ---


DLA DESKRYPTORÓW

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. TWOJA KLASA FEATURIZERA (ADMETDescriptorFeaturizer)
# ==========================================

class ADMETDescriptorFeaturizer:
    def __init__(self, y_column, smiles_col='Drug'):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.feature_names = [
            'MW', 'LogP', 'HBD', 'HBA', 'TPSA',
            'RotatableBonds', 'AromaticRings', 'HeavyAtoms',
            'MolMR', 'FractionCSP3'
        ]
        self.scaler = StandardScaler()

    def __call__(self, df, fit_scaler=True):
        features = []
        labels = []

        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            y = row[self.y_column]
            mol = Chem.MolFromSmiles(smiles)

            if mol:
                desc_vector = [
                    Descriptors.MolWt(mol),
                    Descriptors.MolLogP(mol),
                    Lipinski.NumHDonors(mol),
                    Lipinski.NumHAcceptors(mol),
                    Descriptors.TPSA(mol),
                    Lipinski.NumRotatableBonds(mol),
                    Lipinski.NumAromaticRings(mol),
                    Descriptors.HeavyAtomCount(mol),
                    Descriptors.MolMR(mol),
                    Lipinski.FractionCSP3(mol)
                ]
                features.append(desc_vector)
                labels.append(y)

        features_array = np.array(features)

        if fit_scaler:
            normalized_features = self.scaler.fit_transform(features_array)
        else:
            normalized_features = self.scaler.transform(features_array)

        return normalized_features, np.array(labels).reshape(-1, 1)


# ==========================================
# 3. GŁÓWNY POTOK OBLICZENIOWY
# ==========================================

print("--- START: KLASYFIKACJA NA DESKRYPTORACH ---")

# A. Pobranie danych
data = ADME(name='HIA_Hou')
split = data.get_split()

# B. Generowanie i normalizacja deskryptorów przy użyciu Twojej klasy
featurizer = ADMETDescriptorFeaturizer(y_column='Y')

print("[INFO] Wyliczanie deskryptorów dla zbioru treningowego...")
X_train_np, y_train_np = featurizer(split['train'], fit_scaler=True)

print("[INFO] Wyliczanie deskryptorów dla zbioru testowego...")
X_test_np, y_test_np = featurizer(split['test'], fit_scaler=False)

# C. Konwersja na tensory
X_train_t = torch.FloatTensor(X_train_np)
y_train_t = torch.FloatTensor(y_train_np)
X_test_t = torch.FloatTensor(X_test_np)

# D. Inicjalizacja modelu
# input_dim to teraz 10 (liczba deskryptorów w Twojej liście)
model = STL_NN_Classifier(input_dim=len(featurizer.feature_names))
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# E. Trening
print("\n--- TRENING ---")
model.train()
for epoch in range(101):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"  Epoka {epoch:3d} | Loss: {loss.item():.4f}")

# F. Ewaluacja
model.eval()
with torch.no_grad():
    probs = model(X_test_t).numpy()
    auroc = roc_auc_score(y_test_np, probs)
    print(f"\n[WYNIK] AUROC na deskryptorach: {auroc:.4f}")

print("--- KONIEC ---")

Found local copy...
Loading...
Done!


--- START: KLASYFIKACJA NA DESKRYPTORACH ---
[INFO] Wyliczanie deskryptorów dla zbioru treningowego...
[INFO] Wyliczanie deskryptorów dla zbioru testowego...

--- TRENING ---
  Epoka   0 | Loss: 0.5546
  Epoka  20 | Loss: 0.2459
  Epoka  40 | Loss: 0.1769
  Epoka  60 | Loss: 0.1336
  Epoka  80 | Loss: 0.0844
  Epoka 100 | Loss: 0.0746

[WYNIK] AUROC na deskryptorach: 0.9636
--- KONIEC ---


In [ ]:
# ==========================================
# POTOK DLA CACO-2 (REGRESJA + DESKRYPTORY)
# ==========================================

print("--- START: REGRESJA CACO-2 NA DESKRYPTORACH ---")

# 1. Pobranie danych
data_caco = ADME(name='Caco2_Wang')
split_caco = data_caco.get_split()

# 2. Inicjalizacja Featurizera dla deskryptorów
# Używamy Twojej klasy ADMETDescriptorFeaturizer
featurizer_desc = ADMETDescriptorFeaturizer(y_column='Y')

print("[INFO] Generowanie deskryptorów...")
X_train_desc, y_train_raw = featurizer_desc(split_caco['train'], fit_scaler=True)
X_test_desc, y_test_raw = featurizer_desc(split_caco['test'], fit_scaler=False)

# 3. Normalizacja etykiet (Target Scaling) - kluczowe dla regresji w NN
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train_raw)
# y_test nie fitujemy, tylko transformujemy!
y_test_scaled = y_scaler.transform(y_test_raw)

# 4. Konwersja na tensory
X_train_t = torch.FloatTensor(X_train_desc)
y_train_t = torch.FloatTensor(y_train_scaled)
X_test_t = torch.FloatTensor(X_test_desc)

# 5. Inicjalizacja modelu Regresji
# input_dim = 10 (liczba deskryptorów w Twojej klasie)
model_reg = STL_NN_Regressor(input_dim=len(featurizer_desc.feature_names))
criterion = nn.MSELoss()
optimizer = optim.Adam(model_reg.parameters(), lr=0.001)

# 6. Trening
print("[INFO] Trening modelu regresyjnego...")
model_reg.train()
for epoch in range(101):
    optimizer.zero_grad()
    outputs = model_reg(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"  Epoka {epoch:3d} | MSE Loss: {loss.item():.4f}")

# 7. Ewaluacja i powrót do oryginalnej skali
model_reg.eval()
with torch.no_grad():
    preds_scaled = model_reg(X_test_t).numpy()
    # Odwracamy skalowanie, aby dostać realne wartości logPapp
    preds_final = y_scaler.inverse_transform(preds_scaled)

    # Obliczenie RMSE
    rmse = np.sqrt(mean_squared_error(y_test_raw, preds_final))
    print(f"\n[WYNIK] Finalny RMSE na deskryptorach: {rmse:.4f}")

print("--- KONIEC POTOKU ---")

Found local copy...
Loading...
Done!


--- START: REGRESJA CACO-2 NA DESKRYPTORACH ---
[INFO] Generowanie deskryptorów...
[INFO] Trening modelu regresyjnego...
  Epoka   0 | MSE Loss: 1.3371
  Epoka  20 | MSE Loss: 0.3665
  Epoka  40 | MSE Loss: 0.2857
  Epoka  60 | MSE Loss: 0.2320
  Epoka  80 | MSE Loss: 0.2037
  Epoka 100 | MSE Loss: 0.1740

[WYNIK] Finalny RMSE na deskryptorach: 0.4972
--- KONIEC POTOKU ---


In [ ]:
class MTL_NN_Hybrid(nn.Module):
    def __init__(self, input_dim, reg_tasks, class_tasks, hidden_dim=512):
        """
        reg_tasks: lista nazw zadań regresyjnych (np. ['Lipophilicity', 'Solubility'])
        class_tasks: lista nazw zadań klasyfikacyjnych (np. ['BBB', 'Ames'])
        """
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.reg_tasks = reg_tasks
        self.class_tasks = class_tasks

        # Słowniki głowic dla każdego typu zadania
        self.reg_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in reg_tasks
        })
        self.class_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in class_tasks
        })

    def forward(self, x):
        shared_features = self.encoder(x)
        results = {}

        # Procesowanie zadań regresyjnych
        for task in self.reg_tasks:
            results[task] = self.reg_heads[task](shared_features)

        # Procesowanie zadań klasyfikacyjnych
        for task in self.class_tasks:
            results[task] = torch.sigmoid(self.class_heads[task](shared_features))

        return results